# CHEM 427 — Project 5: Finding the Varnish on a Painting

**Assigned project:** Project 5 — Finding the Varnish on a Painting

**Data source:** Rocha de Oliveira et al. (2023), *Push-broom NIR-HSI scanning of painting reconstruction, inspired by Sandro Botticelli's "Venus"*, Zenodo.

**Date accessed:** November 6, 2026


In [1]:

from pathlib import Path
import re
import numpy as np

# Read the original ENVI header
data_dir = Path("data")
header = (data_dir / "venere.hdr").read_text()

def field(name):
    match = re.search(
        rf"(?im)^\s*{re.escape(name)}\s*=\s*(.+)",
        header
    )
    if match is None:
        raise ValueError(f"Missing field: {name}")
    return match.group(1).strip()

samples = int(field("samples"))
lines = int(field("lines"))
bands = int(field("bands"))
data_type = int(field("data type"))
byte_order = int(field("byte order"))
offset = int(field("header offset"))
interleave = field("interleave").lower()

dtype_map = {
    1: "u1", 2: "i2", 3: "i4",
    4: "f4", 5: "f8", 12: "u2", 13: "u4"
}
dtype = np.dtype(dtype_map[data_type])
if dtype.itemsize > 1:
    dtype = dtype.newbyteorder(
        "<" if byte_order == 0 else ">"
    )

# Load the raw data from the repository
raw = np.fromfile(
    data_dir / "venere.raw",
    dtype=dtype,
    offset=offset
)

assert raw.size == lines * samples * bands

print("Number of rows:", lines)
print("Pixels per row:", samples)
print("Total pixels:", lines * samples)
print("Column names: spectral bands")
print("Number of spectral bands:", bands)
print("Measured variable: raw detector signal")
print("Measured variable units: detector counts")
print("Wavelength units: nm")


Number of rows: 410
Pixels per row: 384
Total pixels: 157440
Column names: spectral bands
Number of spectral bands: 288
Measured variable: raw detector signal
Measured variable units: detector counts
Wavelength units: nm
